# Land cover from a random forest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/07_rf_classification.ipynb)

Notebook 06 ended at a wall: k-means finds groups but cannot name them. This
notebook pays the price that names cost — **labeled examples** — and trains a
random forest to classify land cover. It is the same algorithm family that
notebook 04 used to *sharpen temperatures*; there it predicted a number
(regression), here it predicts a class (classification).

For labels we borrow **[Dynamic World](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1)**,
Google's global 10 m land-cover product. One caveat to carry through the whole
notebook: Dynamic World is itself a *model's* output. Training on it teaches our
forest to imitate that model — not the ground. The honest upgrade is your own
training polygons, and the last section shows where they would plug in.

See this demo for dynamic, month-to-month land-cover change: [Dynamic World Explorer](https://dynamicworld.app/).

## 1 · Install (Colab only)

In [1]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version, which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

## 2 · Authenticate

In [2]:
import os
PROJECT = os.environ.get("EE_PROJECT", "ag-dataplatform")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene — Grumsin and its surroundings

Classification needs **variety**: a frame holding only beech canopy has nothing to
separate. The Grumsin footprint plus a 3 km collar brings in farmland, lakes and
settlements — enough distinct surfaces for the method to have something to find.

In [3]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "wne" / "wne.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = ("https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/main"
           "/docs/aoi/wne/wne.geojson")
    urllib.request.urlretrieve(RAW, "wne.geojson")
    REGION = "wne.geojson"

START, END = "2025-06-01", "2025-09-01"     # one summer, end exclusive

region_geom = aoi.parse({"geojson": REGION})
frame_bbox = list(aoi.frame_bbox_from_region(region_geom, 3000.0))
frame_geom = aoi.parse({"bbox": frame_bbox})
print("frame bbox:", [round(v, 4) for v in frame_bbox], "(footprint + 3 km)")

frame bbox: [13.8245, 52.9479, 13.9687, 53.0241] (footprint + 3 km)


## 4 · One cloud-free summer composite

The animation pipeline keeps only its computed index band, so here we assemble the
multiband input ourselves — from the same three pieces of the package's `Sensor`
object that `collection.build` uses: the source collection, the per-pixel cloud
mask (SCL + s2cloudless, which first needs its probability band joined on), and
the six **harmonised reflectance aliases** from notebook 05 (`blue … swir2`).
The summer median of that is the one image everything below works on.

In [4]:
from gee_animation import products

sensor = products.SENSORS["sentinel2"]
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]

coll = (sensor.collection(ee)
        .filterDate(START, END)
        .filterBounds(frame_geom)
        .filter(ee.Filter.lte(sensor.scene_cloud_property, 70)))   # scene-level pre-filter
coll = sensor.attach_aux(coll, START, END, frame_geom, ee_module=ee)  # join s2cloudless probabilities
coll = (coll.map(lambda img: sensor.mask_clouds(img, ee))     # per-pixel: SCL classes + cloud_prob
            .map(lambda img: sensor.reflectance(img, ee)))    # six harmonised bands, reflectance 0-1
composite = coll.median().clip(frame_geom)

print(coll.size().getInfo(), "cloud-masked scenes in the composite")

20 cloud-masked scenes in the composite


In [5]:
from IPython.display import Image, display

def thumb(image, vis, title):
    url = image.getThumbURL({**vis, "region": frame_geom, "dimensions": 768})
    print(title)
    display(Image(url=url))

thumb(composite, {"bands": ["red", "green", "blue"], "min": 0, "max": 0.17},
      "True colour, summer median " + START[:4])

True colour, summer median 2025


## 5 · Labels, and an honest split

`stratifiedSample` draws the same number of points per class — otherwise the
frame's sea of trees would drown the rare classes and the forest would learn
"when unsure, say trees". The `randomColumn` split sets 30 % aside **before**
training; the forest never sees those points, which is what makes section 7 an
honest check rather than a self-congratulation.

In [6]:
DW_CLASSES = ["water", "trees", "grass", "flooded_vegetation", "crops",
              "shrub_and_scrub", "built", "bare", "snow_and_ice"]
DW_PALETTE = ["419bdf", "397d49", "88b053", "7a87c6", "e49635",
              "dfc35a", "c4281b", "a59b8f", "b39fe1"]

dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterDate(START, END).filterBounds(frame_geom)
      .select("label").mode())                     # each pixel's most frequent class

points = (composite.select(BANDS).addBands(dw)
          .stratifiedSample(numPoints=300, classBand="label", region=frame_geom,
                            scale=20, seed=1, geometries=False)
          .randomColumn(seed=1))
train = points.filter(ee.Filter.lt("random", 0.7))
test = points.filter(ee.Filter.gte("random", 0.7))

rf = ee.Classifier.smileRandomForest(100).train(train, "label", BANDS)
classified = composite.select(BANDS).classify(rf)
print("training points:", train.size().getInfo(), "· held-out test points:", test.size().getInfo())

training points: 1605 · held-out test points: 651


## 6 · The two maps, side by side

In [7]:
vis = {"min": 0, "max": 8, "palette": DW_PALETTE}
thumb(classified, vis, "our random forest (trained on Dynamic World labels)")
thumb(dw.clip(frame_geom), vis, "Dynamic World itself, same summer")

our random forest (trained on Dynamic World labels)


Dynamic World itself, same summer


## 7 · The honesty check

In notebook 04 the check was *OOB RMSE in kelvin*; for a classifier it is the
**confusion matrix** on the held-out points. Read it row by row: row *i*, column
*j* counts test points whose label is class *i* but which the forest called class
*j* — the diagonal is agreement.

And mind what is being measured: this is accuracy **against Dynamic World**, our
label source — not against the ground. A field survey could disagree with both.

In [8]:
cm = test.classify(rf).errorMatrix("label", "classification")
matrix, acc, kappa = cm.getInfo(), cm.accuracy().getInfo(), cm.kappa().getInfo()

present = [i for i, row in enumerate(matrix) if sum(row) or sum(r[i] for r in matrix)]
print(f"{'':>18}" + "".join(f"{DW_CLASSES[j][:7]:>9}" for j in present))
for i in present:
    print(f"{DW_CLASSES[i]:>18}" + "".join(f"{matrix[i][j]:>9}" for j in present))
print(f"\noverall accuracy {acc:.2f} · kappa {kappa:.2f}  (vs Dynamic World, not vs the ground)")

                      water    trees    grass  flooded    crops  shrub_a    built     bare
             water       85        1        0        6        0        0        0        0
             trees        0       68        8        4        1        0        2        0
             grass        0        4       51        1        9        2        9        0
flooded_vegetation        2        2        0       27        1        0        0        0
             crops        0        5       11        0       54       10       13        0
   shrub_and_scrub        0        1       12        1        2       51        3        6
             built        0        6       12        3       11        9       41        8
              bare        0        0        0        0        0       18        0       91

overall accuracy 0.72 · kappa 0.68  (vs Dynamic World, not vs the ground)


## 8 · Regression vs classification — same forest, different question

| | notebook 04 | this notebook |
|---|---|---|
| target | temperature in °C (continuous) | land-cover class (categorical) |
| estimator | `RandomForestRegressor` | `ee.Classifier.smileRandomForest` |
| where it runs | locally, scikit-learn | server-side, Earth Engine |
| honesty check | OOB R² / RMSE, degrade-and-recover | confusion matrix on held-out points |
| failure smell | residual structure | off-diagonal mass between similar classes |

The algorithm is the same ensemble of decision trees in both. What changes is the
**question** — "how warm?" versus "which kind?" — and with it the loss, the
output, and what an honest error statement looks like.

## Make it yours

- Replace the Dynamic World labels with your own: draw polygons per class in the
  [Code Editor](https://code.earthengine.google.com), give each a `label`
  property, `sampleRegions` over them — your accuracy then means accuracy
  against *your* ground truth.
- Drop `swir1`/`swir2` from `BANDS` and retrain: how much does accuracy fall?
  You just measured what SWIR is worth for separating water and built-up.
- Add the month-to-month NDVI range as a feature — phenology separates crops
  from grass far better than any single summer snapshot.
- Swap `REGION` for a city, a coastline, a mining area. The confusion matrix
  will tell you which classes your six bands cannot keep apart there.